# 01. Environment check

Checks that one Kaggle T4 GPU can run the tools used in this project: PyTorch, RAPIDS-singlecell and Geneformer.
Settings used: Accelerator GPU T4 x2 (only the first GPU is used), Internet on.

## 1. GPU and library versions

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use only one T4

import torch
print("torch:", torch.__version__)
print("GPUs visible:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("Compute capability:", torch.cuda.get_device_capability(0))  # native bf16 needs 8.0+

!nvidia-smi
!pip -q install scanpy anndata transformers

## 2. Does the GPU compute?

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch, time

x = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
torch.cuda.synchronize(); t = time.time()
for _ in range(20):
    y = x @ x
torch.cuda.synchronize()
print("fp16 matmul OK in", round(time.time() - t, 3), "s")
print("Memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

## 3. Install RAPIDS-singlecell (restart the kernel afterwards)

In [ ]:
!pip install 'rapids-singlecell-cu13[rapids]' --extra-index-url=https://pypi.nvidia.com 2>&1 | tail -n 15

## 4. GPU preprocessing on a small public dataset

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import scanpy as sc, rapids_singlecell as rsc

print("rapids-singlecell", rsc.__version__)
adata = sc.datasets.pbmc3k()          # small public dataset, needs Internet on
rsc.get.anndata_to_GPU(adata)
rsc.pp.normalize_total(adata)
rsc.pp.log1p(adata)
print("GPU preprocessing OK:", adata.shape)

## 5. Can one T4 run Geneformer V2-104M?

Random token IDs are used on purpose: this measures speed and memory only, not biology.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch, time
from transformers import AutoModelForMaskedLM

model = AutoModelForMaskedLM.from_pretrained(
    "ctheodoris/Geneformer", subfolder="Geneformer-V2-104M",
    torch_dtype=torch.float16,
).cuda().eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Params: {n_params:.0f}M | max positions:", model.config.max_position_embeddings)
print("Weights on GPU (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

seq_len = 4096
for bs in [1, 4, 8, 16]:
    try:
        torch.cuda.reset_peak_memory_stats()
        ids = torch.randint(5, model.config.vocab_size, (bs, seq_len), device="cuda")
        mask = torch.ones_like(ids)
        torch.cuda.synchronize(); t = time.time()
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=mask, output_hidden_states=True)
        torch.cuda.synchronize()
        print(f"batch {bs:>2}: OK | {time.time()-t:.2f}s | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
        del out, ids, mask
        torch.cuda.empty_cache()
    except torch.cuda.OutOfMemoryError:
        print(f"batch {bs:>2}: out of memory")
        torch.cuda.empty_cache()
        break